# Fake News Detection using Natural Language Processing and Machine Learning
### End-to-End Artificial Intelligence & Data Science Notebook

**Project Objective:**
Build and compare Natural Language Processing (NLP) models mapped to classic machine learning estimators—specifically **Logistic Regression**, **Multinomial Naive Bayes**, and **Support Vector Machine (SVM)**—to classify raw news print as either *Real* (1) or *Fake* (0).

## 1. Environment Setup & Dependency Installation
First, we install and import the necessary libraries to perform data cleaning, vectorization, modeling, and plotting.

In [ ]:
# Install NLTK datasets and dependencies inside Google Colab
!pip install pandas numpy scikit-learn nltk matplotlib seaborn fastapi uvicorn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
import nltk
import pickle
import time

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLP resources from NLTK
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

## 2. Load the Dataset
We use the **Fake and Real News Dataset** from Kaggle which contains:
- `True.csv`: Real news articles published around the globe.
- `Fake.csv`: Fabricated, hyper-partisan, and clickbait articles.

If you are running this in Colab, upload both CSV files to the current directory or load them directly.

In [ ]:
# Setup paths (assuming files are hosted locally or uploaded to Google Colab root)
true_path = 'True.csv'
fake_path = 'Fake.csv'

# Check files exist. If not, generate high-fidelity dummy data for immediate execution
import os
for path, state in [(true_path, 'True'), (fake_path, 'Fake')]:
    if not os.path.exists(path):
        print(f"Creating mock {state}.csv for immediate pipeline execution...")
        mock_data = [
            {
                "title": "Bipartisan Accord Reached on Major Federal Funding" if state == 'True' else "STUNNING ALERT: Alien base uncovered beneath historic landmark!",
                "text": "Congress approved spending plans on a broad bipartisan consensus yesterday to fund public departments through next fiscal winter." if state == 'True' else "Leaked military telegrams reveal shocking proof of glowing metal spheres hovering near historical coordinates!",
                "subject": "politics" if state == 'True' else "conspiracy",
                "date": "June 5, 2026"
            }
        ] * 120 # Cohort duplication
        pd.DataFrame(mock_data).to_csv(path, index=False)

# Read data into pandas DataFrames
df_true = pd.read_csv(true_path)
df_fake = pd.read_csv(fake_path)

# Add label column: True/Real = 1, Fake = 0
df_true['label'] = 1
df_fake['label'] = 0

# Merge datasets
df = pd.concat([df_true, df_fake], ignore_index=True)
print(f"Successfully loaded and merged {len(df)} samples ({len(df_true)} True, {len(df_fake)} Fake).")

## 3. Data Preprocessing Module
Before training, text must be cleaned of noisy structures like html, URLs, punctuation, and capitalizations. We write a modular, highly documented preprocessing pipeline.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove brackets
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    # Remove html markup tags
    text = re.sub(r'<.*?>+', '', text)
    # Remove punctuation
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)
    # Remove punctuation/numbers
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize, remove stopwords, and lemmatize
    tokens = word_tokenize(text)
    cleaned_tokens = []
    for token in tokens:
        if token not in stop_words and len(token) > 2:
            lemma = lemmatizer.lemmatize(token, pos='v')
            lemma = lemmatizer.lemmatize(lemma, pos='n')
            cleaned_tokens.append(lemma)
            
    return " ".join(cleaned_tokens)

# Test Preprocessor on a complex mock string
print("Sample Cleaning Outcome:")
print(clean_text("BREAKING! Doctors hate this secret [REVEALED]! Cures flu in 24 hours at http://cure.com/now"))

### Apply Preprocessing to Dataset

In [ ]:
print("Cleaning full texts... (This might take a while on raw dataset)")
df['full_text'] = df['title'] + " " + df['text']
df = df.dropna(subset=['full_text'])

start_time = time.time()
df['cleaned_text'] = df['full_text'].apply(clean_text)
print(f"Completed in {time.time() - start_time:.2f} seconds!")

## 4. Exploratory Data Analysis (EDA)
Let's visualize target balance and analyze text lengths.

In [ ]:
# Plot Target Label Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='label', palette='coolwarm')
plt.title('Distribution of News (0 = Fake, 1 = Real)')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks([0, 1], ['Fake News', 'Real News'])
plt.grid(axis='y', alpha=0.3)
plt.show()

# Calculate Article Length Statistics
df['text_len'] = df['cleaned_text'].apply(lambda x: len(x.split()))
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='text_len', hue='label', element='step', stat='density', common_norm=False, palette='coolwarm')
plt.title('Article Length Comparison (Word Count)')
plt.xlabel('Length in Words')
plt.ylabel('Density')
plt.grid(alpha=0.3)
plt.show()

## 5. Train-Test Split & TF-IDF Extraction
We extract numerical features from our cleaned text using **Term Frequency-Inverse Document Frequency (TF-IDF)** and perform an 80/20 stratified split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# Max Features set to 5000
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Train Vector Shape: {X_train_vec.shape}")
print(f"Test Vector Shape: {X_test_vec.shape}")

## 6. Model Training & Validation Dashboard
We train three models: Logistic Regression, Multinomial Naive Bayes, and Linear Support Vector Classifier, tracking training/testing speeds and metrics.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42, max_iter=2000)
}

results = []
model_cache = {}

for name, clf in models.items():
    print(f"Training {name}...")
    t0 = time.time()
    clf.fit(X_train_vec, y_train)
    train_duration = time.time() - t0
    
    t1 = time.time()
    preds = clf.predict(X_test_vec)
    test_duration = time.time() - t1
    
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary')
    
    model_cache[name] = clf
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "Train Time (s)": train_duration,
        "Test Time (s)": test_duration
    })
    
    print(f"Confusion Matrix for {name}:")
    print(confusion_matrix(y_test, preds))
    print(f"Classification Report for {name}:")
    print(classification_report(y_test, preds))
    print("="*50)

results_df = pd.DataFrame(results)
results_df

## 7. Select & Save the Optimal Pipeline
We pull the model with the highest F1-Score, saving it via `pickle` into binary storage.

In [ ]:
best_row = results_df.loc[results_df['F1-Score'].idxmax()]
best_name = best_row['Model']
best_clf = model_cache[best_name]

print(f"🏆 Selecting {best_name} as optimal final classifier with F1 of {best_row['F1-Score']:.4%}")

os.makedirs('saved_models', exist_ok=True)

with open('saved_models/model.pkl', 'wb') as f:
    pickle.dump(best_clf, f)
with open('saved_models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
    
print("[✓] All pipeline modules serialised successfully to 'saved_models/'")

## 8. Run Production Inference Tests
A self-contained prediction function is declared, running real-time news strings through the vectorizer and pickled model pipelines.

In [ ]:
def predict_news(news_text):
    # Load serialized objects
    with open('saved_models/model.pkl', 'rb') as f:
        loaded_model = pickle.load(f)
    with open('saved_models/vectorizer.pkl', 'rb') as f:
        loaded_vectorizer = pickle.load(f)
        
    # Preprocess
    cleaned = clean_text(news_text)
    vec_text = loaded_vectorizer.transform([cleaned])
    
    # Eval
    pred_idx = loaded_model.predict(vec_text)[0]
    label_out = "Real News" if pred_idx == 1 else "Fake News"
    
    return label_out

# Test predictor on standard headlines
print("Prediction test 1:")
print(predict_news("Secret scientific test shows chocolate bars are descending from outer clouds!"))

print("\nPrediction test 2:")
print(predict_news("The Senate reached a broad bipartisan agreement to establish national education benchmarks last night."))